# Novelty Aspect — Dysbiosis ↔ Hippo-Activated CRC Tumors (Overlap + Direction + Enrichment)

## What we are testing (core idea)
If dysbiosis is linked to colorectal cancer progression, then genes altered in IBD (dysbiosis/inflammation)
should overlap with genes associated with Hippo/YAP activation in CRC tumors.

## Inputs (must exist from previous notebooks)
IBD:
- `results/de/ibd/IBD_DEG_IBD_vs_Control.csv`

TCGA:
- `results/de/tcga/TCGA_DEG_HippoHigh_vs_HippoLow.csv`

Outputs:
- overlap tables (same direction / opposite direction)
- a "core signature" list
- simple visualizations
- enrichment results (optional)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/GeneticCodonShared/Hippo_Dysbiosis"

IBD_DEG  = f"{BASE}/results/de/ibd/IBD_DEG_IBD_vs_Control.csv"
TCGA_DEG = f"{BASE}/results/de/tcga/TCGA_DEG_HippoHigh_vs_HippoLow.csv"

# optional for visualization
IBD_LOG  = f"{BASE}/data_processed/ibd/ibd_log2_tpm.csv"
TCGA_LOG = f"{BASE}/data_processed/tcga/tcga_log2_rsem.csv"

OUT = f"{BASE}/results/novelty"
FIG = f"{BASE}/results/figures/novelty"

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

print("IBD DEG :", IBD_DEG)
print("TCGA DEG:", TCGA_DEG)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#load tables and sanity checks

ibd = pd.read_csv(IBD_DEG)
tcga = pd.read_csv(TCGA_DEG)

print("IBD DEG shape:", ibd.shape)
print("TCGA DEG shape:", tcga.shape)

print("IBD columns:", ibd.columns.tolist())
print("TCGA columns:", tcga.columns.tolist())

# Expected columns:
# IBD: gene, logFC_IBD_minus_Control, padj_fdr
# TCGA: gene, logFC_Hippo_high_minus_Hippo_low, padj_fdr

display(ibd.head(3))
display(tcga.head(3))


**Filter Significant DEGs**

In [ ]:
def filter_deg(df, logfc_col, fdr_col="padj_fdr", fdr=0.05, abs_logfc=0.5):
    df = df.copy()
    df[fdr_col] = pd.to_numeric(df[fdr_col], errors="coerce").fillna(1.0)
    df[logfc_col] = pd.to_numeric(df[logfc_col], errors="coerce")
    keep = (df[fdr_col] < fdr) & (df[logfc_col].abs() >= abs_logfc)
    return df.loc[keep].copy()

IBD_LOGFC  = "logFC_IBD_minus_Control"
TCGA_LOGFC = "logFC_Hippo_high_minus_Hippo_low"

ibd_sig  = filter_deg(ibd,  IBD_LOGFC,  fdr=0.05, abs_logfc=0.5)
tcga_sig = filter_deg(tcga, TCGA_LOGFC, fdr=0.05, abs_logfc=0.5)

print("IBD sig :", ibd_sig.shape)
print("TCGA sig:", tcga_sig.shape)

ibd_sig.to_csv(f"{OUT}/IBD_sig_genes.csv", index=False)
tcga_sig.to_csv(f"{OUT}/TCGA_sig_genes.csv", index=False)


**Overlap identification between conditions (same or opposite direction)**

In [ ]:
m = ibd_sig[["gene", IBD_LOGFC, "padj_fdr"]].merge(
    tcga_sig[["gene", TCGA_LOGFC, "padj_fdr"]],
    on="gene",
    suffixes=("_IBD", "_TCGA"),
    how="inner"
)

m["direction_IBD"]  = np.sign(m[IBD_LOGFC])
m["direction_TCGA"] = np.sign(m[TCGA_LOGFC])

m["direction_match"] = np.where(m["direction_IBD"] == m["direction_TCGA"],
                                "same_direction", "opposite_direction")

print("Total overlap:", m.shape[0])
print(m["direction_match"].value_counts())

m.to_csv(f"{OUT}/overlap_IBD_TCGA_all.csv", index=False)
m[m["direction_match"]=="same_direction"].to_csv(f"{OUT}/overlap_IBD_TCGA_same_direction.csv", index=False)
m[m["direction_match"]=="opposite_direction"].to_csv(f"{OUT}/overlap_IBD_TCGA_opposite_direction.csv", index=False)

display(m.head(10))


**Overlap-volcano plot**


In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(m[IBD_LOGFC], m[TCGA_LOGFC], alpha=0.6)

plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")

plt.title("Overlap genes: IBD logFC vs TCGA (Hippo-high vs Hippo-low) logFC")
plt.xlabel("IBD log2FC (IBD - Control)")
plt.ylabel("TCGA log2FC (Hippo-high - Hippo-low)")
plt.tight_layout()
plt.savefig(f"{FIG}/overlap_scatter_IBD_vs_TCGA.png", dpi=200)
plt.show()


**Build a “core signature” list**

Core = overlap genes with same direction + strongest evidence.

In [ ]:
same = m[m["direction_match"]=="same_direction"].copy()

# rank by combined significance (simple scoring)
same["score"] = -np.log10(same["padj_fdr_IBD"] + 1e-300) + -np.log10(same["padj_fdr_TCGA"] + 1e-300)
same = same.sort_values("score", ascending=False)

core_genes = same["gene"].tolist()
pd.DataFrame({"core_signature_genes": core_genes}).to_csv(f"{OUT}/core_signature_genes.csv", index=False)

print("Core signature size:", len(core_genes))
display(same.head(15))


## **Heatmap**

In [ ]:
topN = 20
top_genes = core_genes[:topN]

if len(top_genes) < 3:
    print("Core list too small. Lower abs_logfc threshold (e.g., 0.5) in filtering step.")
else:
    ibd_log  = pd.read_csv(IBD_LOG, index_col=0)
    tcga_log = pd.read_csv(TCGA_LOG, index_col=0)

    # keep only genes present
    top_genes = [g for g in top_genes if (g in ibd_log.index and g in tcga_log.index)]
    print("Genes used for heatmap-like plot:", len(top_genes))

    def z(expr):
        return expr.sub(expr.mean(1), axis=0).div(expr.std(1).replace(0, np.nan), axis=0).fillna(0)

    ibd_v  = z(ibd_log.loc[top_genes])
    tcga_v = z(tcga_log.loc[top_genes])

    plt.figure(figsize=(10,5))
    plt.imshow(ibd_v.values, aspect="auto")
    plt.yticks(range(len(top_genes)), top_genes)
    plt.title("IBD (z-score) — Top overlap genes")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{FIG}/IBD_top_overlap_heatmaplike.png", dpi=200)
    plt.show()

    plt.figure(figsize=(10,5))
    plt.imshow(tcga_v.values, aspect="auto")
    plt.yticks(range(len(top_genes)), top_genes)
    plt.title("TCGA (z-score) — Top overlap genes")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{FIG}/TCGA_top_overlap_heatmaplike.png", dpi=200)
    plt.show()


# **Enrichment Analysis**

In [ ]:
try:
    import gseapy as gp
except ImportError:
    !pip -q install gseapy
    import gseapy as gp

if len(core_genes) >= 5:
    enr = gp.enrichr(
        gene_list=core_genes,
        gene_sets=["GO_Biological_Process_2021", "KEGG_2021_Human", "MSigDB_Hallmark_2020"],
        organism="Human",
        outdir=None
    )
    res = enr.results.sort_values("Adjusted P-value")
    res.to_csv(f"{OUT}/core_signature_enrichment.csv", index=False)
    print("Saved ->", f"{OUT}/core_signature_enrichment.csv")
    display(res.head(20))
else:
    print("Not enough core genes for enrichment. Lower thresholds in filter step.")


In [ ]:
```python
import pandas as pd
import matplotlib.pyplot as plt
import os

try:
    import gseapy as gp
except ImportError:
    !pip -q install gseapy
    import gseapy as gp

# Ensure OUT and FIG are defined, assuming previous cells have been run
# If running this cell independently, uncomment and define them:
# BASE = "/content/drive/MyDrive/GeneticCodonShared/Hippo_Dysbiosis"
# OUT = f"{BASE}/results/novelty"
# FIG = f"{BASE}/results/figures/novelty"

enrichment_filepath = f"{OUT}/core_signature_enrichment.csv"

if os.path.exists(enrichment_filepath):
    res = pd.read_csv(enrichment_filepath)

    if not res.empty:
        # Sort by Adjusted P-value for plotting top terms
        res_sorted = res.sort_values(by="Adjusted P-value", ascending=True)

        # Define top N terms to plot
        num_top_terms = 15

        # Bar plot
        print("Generating bar plot...")
        plt.figure(figsize=(10, 8))
        ax = gp.plot.barplot(enr_res=res_sorted, title="Top Enriched Pathways (Bar Plot)",
                             ofname=None, cutoff=0.05, top_term=num_top_terms)
        plt.tight_layout()
        bar_plot_path = f"{FIG}/enrichment_barplot.png"
        plt.savefig(bar_plot_path, dpi=300)
        plt.show()
        print(f"Enrichment bar plot saved to {bar_plot_path}")

        # Dot plot
        print("Generating dot plot...")
        plt.figure(figsize=(10, 8))
        ax = gp.plot.dotplot(enr_res=res_sorted, title="Top Enriched Pathways (Dot Plot)",
                             ofname=None, cutoff=0.05, top_term=num_top_terms, size=10, legend_n=5)
        plt.tight_layout()
        dot_plot_path = f"{FIG}/enrichment_dotplot.png"
        plt.savefig(dot_plot_path, dpi=300)
        plt.show()
        print(f"Enrichment dot plot saved to {dot_plot_path}")

    else:
        print("Enrichment results file is empty, no plots generated.")
else:
    print("Enrichment results file not found. Please ensure the enrichment analysis was run successfully.")
```